<a href="https://colab.research.google.com/github/TPShipilova/Cyberphys/blob/main/1lab/%5B%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D1%84%D0%B8%D0%B7%5D%D0%A8%D0%B8%D0%BF%D0%B8%D0%BB%D0%BE%D0%B2%D0%B0_406_1%D0%9B%D0%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm opencv-python-headless xmltodict pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:0000:01


In [ ]:
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

In [ ]:
import os, glob, cv2, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
IMG_DIR = "/kaggle/input/datasets/andrewmvd/face-mask-detection/images"
ANN_DIR = "/kaggle/input/datasets/andrewmvd/face-mask-detection/annotations"
MASK_DIR = "/kaggle/output/masks"
os.makedirs(MASK_DIR, exist_ok=True)

# Маппинг классов
CLASS_NAMES = ['with_mask', 'without_mask', 'mask_weared_incorrect']
CLASS_MAP = {name: i+1 for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES) + 1

print(f"Class mapping: {CLASS_MAP}")

xml_files = sorted(glob.glob(os.path.join(ANN_DIR, '*.xml')))
for xml_path in xml_files:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)

    mask = np.zeros((h, w), dtype=np.uint8)

    for obj in root.findall('object'):
        cls_name = obj.find('name').text
        bbox = obj.find('bndbox')
        if cls_name in CLASS_MAP:
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)
            mask[ymin:ymax, xmin:xmax] = CLASS_MAP[cls_name]

    cv2.imwrite(os.path.join(MASK_DIR, filename.replace('.jpg', '.png').replace('.png', '.png')), mask)

print(f"Сгенерировано масок: {len(os.listdir(MASK_DIR))}")

Class mapping: {'with_mask': 1, 'without_mask': 2, 'mask_weared_incorrect': 3}
Сгенерировано масок: 853


In [ ]:
class MaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, limit=None):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, '*.*')))
        self.mask_paths = sorted(glob.glob(os.path.join(mask_dir, '*.png')))
        self.transform = transform
        if limit: self.img_paths = self.img_paths[:limit]; self.mask_paths = self.mask_paths[:limit]

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.img_paths[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented['image'], augmented['mask']

        img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        mask = torch.from_numpy(mask.astype(np.int64))
        return img, mask

# Метрики IoU, Dice, Pixel Accuracy
def compute_metrics(pred, target, num_classes, ignore_index=0):
    pred = torch.argmax(pred, dim=1)
    ious, dices, accs = [], [], []
    for cls in range(1, num_classes):
        tp = ((pred == cls) & (target == cls)).sum().float()
        fp = ((pred == cls) & (target != cls)).sum().float()
        fn = ((pred != cls) & (target == cls)).sum().float()
        tn = ((pred != cls) & (target != cls)).sum().float()

        iou = tp / (tp + fp + fn + 1e-6)
        dice = 2*tp / (2*tp + fp + fn + 1e-6)
        acc = (tp + tn) / (tp + tn + fp + fn + 1e-6)

        ious.append(iou.item()); dices.append(dice.item()); accs.append(acc.item())
    return np.mean(ious), np.mean(dices), np.mean(accs)

# Разделение на train/val
from sklearn.model_selection import train_test_split
all_imgs = sorted(glob.glob(os.path.join(IMG_DIR, '*.*')))
all_masks = sorted(glob.glob(os.path.join(MASK_DIR, '*.png')))
train_imgs, val_imgs, train_masks, val_masks = train_test_split(all_imgs, all_masks, test_size=0.2, random_state=42)

class PathDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths; self.mask_paths = mask_paths; self.transform = transform
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.img_paths[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if self.transform:
            aug = self.transform(image=img, mask=mask); img, mask = aug['image'], aug['mask']
        img = torch.from_numpy(img.transpose(2,0,1)).float()/255.0
        mask = torch.from_numpy(mask.astype(np.int64))
        return img, mask

# Аугментации (для baseline пустые)
train_transform = A.Compose([A.Resize(256, 256), A.Normalize()])
val_transform = A.Compose([A.Resize(256, 256), A.Normalize()])

train_dataset = PathDataset(train_imgs, train_masks, train_transform)
val_dataset = PathDataset(val_imgs, val_masks, val_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# 1. Выбор начальных условий

a) Обоснование датасета:

Датасет по детекции масок решает актуальную задачу компьютерного зрения в области общественного здравоохранения и безопасности. Автоматический контроль соблюдения масочного режима востребован в умных зданиях, аэропортах, медицинских учреждениях и системах видеонаблюдения. Уникальность набора в наличии трех классов (маска надета правильно, маска надета неправильно, без маски), что позволяет обучить модель различать не только наличие аксессуара, но и корректность его использования.

b) Метрики качества:

- Mean IoU (Intersection over Union): Стандарт для сегментации. Учитывает пересечение и объединение предсказанных и истинных областей, устойчив к дисбалансу классов.
- Mean Dice Coefficient: Аналог IoU, но сильнее штрафует за ложноположительные срабатывания. Чувствителен к точности границ.
- Pixel Accuracy: Доля правильно классифицированных пикселей. Интуитивно понятна, но может вводить в заблуждение при доминировании фона.

# 2. Создание бейзлайна (CNN + Transformer)

In [ ]:
# Базовые модели
# CNN: U-Net + ResNet34
model_cnn = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                     in_channels=3, classes=NUM_CLASSES, activation=None)
# Transformer: U-Net + MiT-B0 (SegFormer encoder)
model_transformer = smp.Unet(encoder_name="mit_b0", encoder_weights="imagenet",
                             in_channels=3, classes=NUM_CLASSES, activation=None)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            all_preds.append(logits.cpu())
            all_targets.append(masks.cpu())
    preds = torch.cat(all_preds, dim=0)
    tgts = torch.cat(all_targets, dim=0)
    iou, dice, acc = compute_metrics(preds, tgts, NUM_CLASSES)
    return iou, dice, acc

In [ ]:
# Обучение baseline
def run_baseline(model, name, epochs=10, lr=1e-3):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"iou":[], "dice":[], "acc":[], "val_loss":[]}

    for ep in range(epochs):
        train_loss = train_epoch(model, train_loader, opt, loss_fn)
        val_iou, val_dice, val_acc = evaluate(model, val_loader)
        history["iou"].append(val_iou); history["dice"].append(val_dice); history["acc"].append(val_acc)
        print(f"[{name}] Ep {ep+1}/{epochs} | IoU:{val_iou:.3f} | Dice:{val_dice:.3f} | Acc:{val_acc:.3f}")
    return history, val_iou, val_dice, val_acc


In [ ]:
print("Training CNN Baseline")
hist_cnn, iou_cnn, dice_cnn, acc_cnn = run_baseline(model_cnn, "CNN-ResNet34")

Training CNN Baseline
[CNN-ResNet34] Ep 1/10 | IoU:0.138 | Dice:0.195 | Acc:0.981
[CNN-ResNet34] Ep 2/10 | IoU:0.206 | Dice:0.255 | Acc:0.986
[CNN-ResNet34] Ep 3/10 | IoU:0.203 | Dice:0.252 | Acc:0.985
[CNN-ResNet34] Ep 4/10 | IoU:0.291 | Dice:0.380 | Acc:0.988
[CNN-ResNet34] Ep 5/10 | IoU:0.389 | Dice:0.489 | Acc:0.989
[CNN-ResNet34] Ep 6/10 | IoU:0.382 | Dice:0.483 | Acc:0.988
[CNN-ResNet34] Ep 7/10 | IoU:0.300 | Dice:0.404 | Acc:0.983
[CNN-ResNet34] Ep 8/10 | IoU:0.450 | Dice:0.537 | Acc:0.991
[CNN-ResNet34] Ep 9/10 | IoU:0.450 | Dice:0.536 | Acc:0.991
[CNN-ResNet34] Ep 10/10 | IoU:0.437 | Dice:0.526 | Acc:0.990


In [ ]:
print("Training Transformer Baseline")
hist_trans, iou_trans, dice_trans, acc_trans = run_baseline(model_transformer, "Transformer-MiT")

Training Transformer Baseline
[Transformer-MiT] Ep 1/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 2/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 3/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 4/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 5/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 6/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 7/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Transformer-MiT] Ep 8/10 | IoU:0.047 | Dice:0.083 | Acc:0.973
[Transformer-MiT] Ep 9/10 | IoU:0.055 | Dice:0.095 | Acc:0.975
[Transformer-MiT] Ep 10/10 | IoU:0.102 | Dice:0.156 | Acc:0.978


Выводы

| Модель | IoU (final) | Dice (final) | Pixel Acc (final) | Сходимость |
|-|-|-|-|-|
| CNN-ResNet34 | 0.437 | 0.526 | 0.990 | Стабильная, с 4-й эпохи |
| Transformer-MiT | 0.102 | 0.156 | 0.978 | Задержка старта, медленная |

Выводы:

- CNN-ResNet34 показал преимущество на малых данных: За 10 эпох модель достигла IoU=0.437, демонстрируя быструю сходимость благодаря предобучению на ImageNet и индуктивным смещениям сверток (локальность, инвариантность к сдвигу).
- Transformer требует больше ресурсов для обучения: МiТ-B0 показал нулевые IoU/Dice в первые 7 эпох — типичное поведение для ViT-архитектур, которым нужно больше эпох для «понимания» пространственных зависимостей. На 853 изображениях этого недостаточно.
- Значения Pixel Accuracy 0.97–0.99 отражают доминирование фона в изображениях. Реальное качество сегментации объектов корректно отражают только IoU и Dice.
- Нестабильность обучения CNN: Наблюдаемый спад метрик на 7-й эпохе (IoU: было 0.450, стало 0.300) указывает на возможное переобучение или слишком высокий learning rate. В пункте 3 это будет исправлено планировщиком и аугментациями.
- Для датасетов <1000 изображений в задачах сегментации целесообразно начинать с CNN-бейзлайна (U-Net + ResNet). Transformer-архитектуры стоит подключать после отладки пайплайна и при наличии вычислительных ресурсов для обучения >50 эпох.

# 3. Улучшение бейзлайна

a) Гипотезы:

Аугментации: Добавить HorizontalFlip, RandomRotate90, ColorJitter для увеличения разнообразия и борьбы с переобучением.

Функция потерь: Заменить CrossEntropyLoss на DiceLoss (лучше работает с несбалансированными классами и маленькими объектами).

Планировщик LR: CosineAnnealingLR для более стабильной сходимости.

Архитектура: DeepLabV3Plus с mit_b0 (лучше захватывает контекст).

Кроме того, увеличим количество эпох обучения в 2 раза.

b-f) Проверка и обучение улучшенной версии:

In [ ]:
# Улучшенные аугментации
train_aug = A.Compose([
    A.Resize(256, 256), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5), A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.Normalize()
])

train_dataset_imp = PathDataset(train_imgs, train_masks, train_aug)
train_loader_imp = DataLoader(train_dataset_imp, batch_size=8, shuffle=True, num_workers=2)

model_improved = smp.DeepLabV3Plus(encoder_name="mit_b0", encoder_weights="imagenet",
                                   in_channels=3, classes=NUM_CLASSES, activation=None)


In [ ]:
# Сверточная сеть
def run_improved(model, epochs=20, lr=5e-4):
    model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = DiceLoss(mode='multiclass')

    best_iou = 0
    history = {"iou": [], "dice": [], "acc": []}

    for ep in range(epochs):
        model.train()
        for imgs, masks in train_loader_imp:
            imgs, masks = imgs.to(device), masks.to(device).long()
            opt.zero_grad()
            logits = model(imgs)
            loss = loss_fn(logits, masks)
            loss.backward()
            opt.step()
        scheduler.step()

        val_iou, val_dice, val_acc = evaluate(model, val_loader)
        history["iou"].append(val_iou)
        history["dice"].append(val_dice)
        history["acc"].append(val_acc)
        print(f"[Improved DLv3+] Ep {ep+1}/{epochs} | IoU:{val_iou:.3f} | Dice:{val_dice:.3f} | Acc:{val_acc:.3f}")

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), 'best_improved.pth')

    return history, val_iou, val_dice, val_acc

print("Training Improved Baseline...")
hist_imp, iou_imp, dice_imp, acc_imp = run_improved(model_improved)

Training Improved Baseline...
[Improved DLv3+] Ep 1/20 | IoU:0.052 | Dice:0.089 | Acc:0.936
[Improved DLv3+] Ep 2/20 | IoU:0.036 | Dice:0.064 | Acc:0.966
[Improved DLv3+] Ep 3/20 | IoU:0.045 | Dice:0.080 | Acc:0.914
[Improved DLv3+] Ep 4/20 | IoU:0.054 | Dice:0.094 | Acc:0.926
[Improved DLv3+] Ep 5/20 | IoU:0.073 | Dice:0.129 | Acc:0.941
[Improved DLv3+] Ep 6/20 | IoU:0.090 | Dice:0.156 | Acc:0.949
[Improved DLv3+] Ep 7/20 | IoU:0.098 | Dice:0.170 | Acc:0.937
[Improved DLv3+] Ep 8/20 | IoU:0.075 | Dice:0.128 | Acc:0.953
[Improved DLv3+] Ep 9/20 | IoU:0.088 | Dice:0.155 | Acc:0.937
[Improved DLv3+] Ep 10/20 | IoU:0.091 | Dice:0.159 | Acc:0.966
[Improved DLv3+] Ep 11/20 | IoU:0.122 | Dice:0.206 | Acc:0.960
[Improved DLv3+] Ep 12/20 | IoU:0.100 | Dice:0.171 | Acc:0.952
[Improved DLv3+] Ep 13/20 | IoU:0.104 | Dice:0.179 | Acc:0.953
[Improved DLv3+] Ep 14/20 | IoU:0.117 | Dice:0.199 | Acc:0.960
[Improved DLv3+] Ep 15/20 | IoU:0.123 | Dice:0.208 | Acc:0.965
[Improved DLv3+] Ep 16/20 | IoU:0.

In [ ]:
from segmentation_models_pytorch.losses import DiceLoss

# Трансформерная модель
model_segformer = smp.Segformer(
    encoder_name="mit_b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

def run_improved_transformer(model, name="SegFormer", epochs=20, lr=5e-4):
    model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = DiceLoss(mode='multiclass')

    best_iou = 0
    history = {"iou": [], "dice": [], "acc": []}

    for ep in range(epochs):
        model.train()
        for imgs, masks in train_loader_imp:
            imgs, masks = imgs.to(device), masks.to(device).long()
            opt.zero_grad()
            logits = model(imgs)
            loss = loss_fn(logits, masks)
            loss.backward()
            opt.step()
        scheduler.step()

        val_iou, val_dice, val_acc = evaluate(model, val_loader)
        history["iou"].append(val_iou)
        history["dice"].append(val_dice)
        history["acc"].append(val_acc)
        print(f"[{name} Improved] Ep {ep+1}/{epochs} | IoU:{val_iou:.3f} | Dice:{val_dice:.3f} | Acc:{val_acc:.3f}")
        if val_iou > best_iou: best_iou = val_iou
    return history, best_iou

print("Training Improved SegFormer")
hist_seg, iou_seg = run_improved_transformer(model_segformer, "SegFormer")

Training Improved SegFormer
[SegFormer Improved] Ep 1/20 | IoU:0.000 | Dice:0.000 | Acc:0.973
[SegFormer Improved] Ep 2/20 | IoU:0.051 | Dice:0.088 | Acc:0.923
[SegFormer Improved] Ep 3/20 | IoU:0.046 | Dice:0.080 | Acc:0.957
[SegFormer Improved] Ep 4/20 | IoU:0.044 | Dice:0.077 | Acc:0.959
[SegFormer Improved] Ep 5/20 | IoU:0.043 | Dice:0.080 | Acc:0.842
[SegFormer Improved] Ep 6/20 | IoU:0.057 | Dice:0.104 | Acc:0.949
[SegFormer Improved] Ep 7/20 | IoU:0.087 | Dice:0.152 | Acc:0.950
[SegFormer Improved] Ep 8/20 | IoU:0.110 | Dice:0.186 | Acc:0.949
[SegFormer Improved] Ep 9/20 | IoU:0.148 | Dice:0.239 | Acc:0.959
[SegFormer Improved] Ep 10/20 | IoU:0.146 | Dice:0.234 | Acc:0.962
[SegFormer Improved] Ep 11/20 | IoU:0.179 | Dice:0.279 | Acc:0.969
[SegFormer Improved] Ep 12/20 | IoU:0.178 | Dice:0.280 | Acc:0.973
[SegFormer Improved] Ep 13/20 | IoU:0.174 | Dice:0.274 | Acc:0.968
[SegFormer Improved] Ep 14/20 | IoU:0.184 | Dice:0.285 | Acc:0.971
[SegFormer Improved] Ep 15/20 | IoU:0.213 |

Выводы:

| Модель | Режим | Эпох | IoU | Dice | Acc | Примечание |
|-|-|-|-|-|-|-|
| CNN-ResNet34 | Baseline | 10 | 0.437 | 0.526 | 0.990 | Быстрая сходимость, стабильно |
| CNN-ResNet34 | Improved (DLv3+) | 20 | 0.116 | 0.196 | 0.957 | Деградация, возможное переобучение |
| Transformer-MiT | Baseline | 10 | 0.102 | 0.156 | 0.978 | Медленный старт, не успел сойтись |
| SegFormer | Improved | 20 | 0.255 | 0.368 | 0.977 | Улучшение в 2.5 раза, но ниже CNN |

Оценка качества улучшенных моделей:

- DeepLabV3+ (Improved): IoU=0.116, Dice=0.196. Модель показала нестабильную динамику с колебаниями метрик и итоговой деградацией относительно простого бейзлайна.
- SegFormer (Improved): IoU=0.255, Dice=0.368. Демонстрирует устойчивый рост после 8-й эпохи, превзошёл базовый Transformer в 2.5 раза по IoU.

Итоговые выводы:
- Не все улучшения универсальны. Аугментации и DiceLoss, полезные для Transformer, могут дестабилизировать обучение CNN на малых выборках, если не скорректирован learning rate и стратегия регуляризации.
- Сложность архитектуры должна соответствовать объёму данных. DeepLabV3+ с предобученным MiT-энкодером имеет больше параметров, чем простой U-Net+ResNet34, и склонен к переобучению при 853 изображениях.
- Transformer-архитектуры требуют «разогрева». Улучшенный SegFormer начал показывать рост только после 8–10 эпох, что подтверждает необходимость более длительного обучения и/или warmup-фазы для внимания.

# 4. Имплементация моделей с нуля (Custom U-Net)

In [ ]:
# Самостоятельная имплементация U-Net
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class CustomUNet(nn.Module):
    def __init__(self, in_channels=3, classes=4):
        super().__init__()
        # ENCODER
        self.down1 = ConvBlock(in_channels, 64)
        self.down2 = ConvBlock(64, 128)
        self.down3 = ConvBlock(128, 256)
        self.down4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

        # BOTTLENECK
        self.bottleneck = ConvBlock(512, 1024)

        # DECODER (up_channels + skip_channels)
        self.up4_t = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.block4 = ConvBlock(1024, 512)  # 512(up) + 512(skip) = 1024

        self.up3_t = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.block3 = ConvBlock(512, 256)   # 256(up) + 256(skip) = 512

        self.up2_t = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.block2 = ConvBlock(256, 128)   # 128(up) + 128(skip) = 256

        self.up1_t = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.block1 = ConvBlock(128, 64)    # 64(up)  + 64(skip)  = 128

        self.final = nn.Conv2d(64, classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))

        b = self.bottleneck(self.pool(d4))

        u4 = self.up4_t(b)
        u4 = torch.cat([u4, d4], dim=1)
        u4 = self.block4(u4)

        u3 = self.up3_t(u4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.block3(u3)

        u2 = self.up2_t(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.block2(u2)

        u1 = self.up1_t(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.block1(u1)

        return self.final(u1)

In [ ]:
# Самостоятельная имплементация трансформера
import math

class PatchEmbed(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x + self.pos_embed

class CustomViTUNet(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=3, classes=4, embed_dim=384, depth=2, heads=8, mlp_ratio=4):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        self.hp = img_size // patch_size  # = 16

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=heads, dim_feedforward=embed_dim*mlp_ratio, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # Декодер: 4 шага апсемплинга для восстановления 16x16 -> 256x256
        self.up1 = nn.ConvTranspose2d(embed_dim, 256, 2, stride=2)
        self.conv1 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())

        self.up3 = nn.ConvTranspose2d(32, 32, 2, stride=2)
        self.conv3 = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())

        self.up4 = nn.ConvTranspose2d(16, 16, 2, stride=2)
        self.conv4 = nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())

        self.final = nn.Conv2d(16, classes, 1)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = self.transformer(x)
        x = x.transpose(1, 2).view(B, -1, self.hp, self.hp)

        x = self.conv1(self.up1(x))
        x = self.conv2(self.up2(x))
        x = self.conv3(self.up3(x))
        x = self.conv4(self.up4(x))
        return self.final(x)

model_custom_transformer = CustomViTUNet(img_size=256, patch_size=16, in_ch=3, classes=NUM_CLASSES,
                                         embed_dim=384, depth=2, heads=8)

In [ ]:
# Обучение и оценка Custom Baseline
# Используем тот же train_loader (без аугментаций)
model_custom = CustomUNet(in_channels=3, classes=NUM_CLASSES).to(device)
opt_c = optim.Adam(model_custom.parameters(), lr=1e-3)
loss_fn_c = nn.CrossEntropyLoss()

print("Training Custom U-Net (Baseline)")
for ep in range(10):
    model_custom.train()
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        opt_c.zero_grad()
        out = model_custom(imgs)
        loss = loss_fn_c(out, masks.long()) # .long() гарантирует правильный тип для CE
        loss.backward()
        opt_c.step()

    iou_c, dice_c, acc_c = evaluate(model_custom, val_loader)
    print(f"[Custom Baseline] Ep {ep+1}/10 | IoU:{iou_c:.3f} | Dice:{dice_c:.3f} | Acc:{acc_c:.3f}")

custom_b_res = (iou_c, dice_c, acc_c)

Training Custom U-Net (Baseline)
[Custom Baseline] Ep 1/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Baseline] Ep 2/10 | IoU:0.017 | Dice:0.033 | Acc:0.952
[Custom Baseline] Ep 3/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Baseline] Ep 4/10 | IoU:0.010 | Dice:0.020 | Acc:0.973
[Custom Baseline] Ep 5/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Baseline] Ep 6/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Baseline] Ep 7/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Baseline] Ep 8/10 | IoU:0.036 | Dice:0.065 | Acc:0.975
[Custom Baseline] Ep 9/10 | IoU:0.002 | Dice:0.004 | Acc:0.973
[Custom Baseline] Ep 10/10 | IoU:0.000 | Dice:0.000 | Acc:0.973


In [ ]:
# Обучение кастомного Трансформера (Базовый уровень)
print("Training Custom Transformer (Baseline)")

# Инициализация модели (использую параметры, оптимальные для Colab)
model_custom_trans_base = CustomViTUNet(
    img_size=256, patch_size=16, in_ch=3, classes=NUM_CLASSES,
    embed_dim=384, depth=2, heads=8
).to(device)

# Параметры обучения
opt_trans_base = optim.Adam(model_custom_trans_base.parameters(), lr=1e-3)
loss_fn_trans_base = nn.CrossEntropyLoss()

for ep in range(10):
    model_custom_trans_base.train()
    for imgs, masks in train_loader:  # Базовый даталоадер (без аугментаций)
        imgs, masks = imgs.to(device), masks.to(device).long()
        opt_trans_base.zero_grad()
        logits = model_custom_trans_base(imgs)
        loss = loss_fn_trans_base(logits, masks)
        loss.backward()
        opt_trans_base.step()

    iou_t_base, dice_t_base, acc_t_base = evaluate(model_custom_trans_base, val_loader)
    print(f"[Custom Trans Baseline] Ep {ep+1}/10 | IoU:{iou_t_base:.3f} | Dice:{dice_t_base:.3f} | Acc:{acc_t_base:.3f}")

custom_trans_base_res = (iou_t_base, dice_t_base, acc_t_base)

Training Custom Transformer (Baseline)
[Custom Trans Baseline] Ep 1/10 | IoU:0.000 | Dice:0.001 | Acc:0.972
[Custom Trans Baseline] Ep 2/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 3/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 4/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 5/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 6/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 7/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 8/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 9/10 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Trans Baseline] Ep 10/10 | IoU:0.000 | Dice:0.000 | Acc:0.973


Выводы по базовому обучению кастомных моделей

| Модель | Режим | Эпохи | IoU (max) | Dice (max) | Acc (final) | Статус обучения |  
|-|-|-|-|-|-|-|
| Custom U-Net |Baseline | 10 | 0.036 | 0.065 | 0.973 | Нестабильное |
| Custom Transformer | Baseline | 10 | 0.000 | 0.000 | 0.973 | Не обучилась |
| CNN-ResNet34 | Lib Baseline | 10 | 0.437 | 0.526 | 0.990 | Успешное |
| Transformer-MiT | Lib Baseline | 10 | 0.102 | 0.156 | 0.978 | Медленное, но есть прогресс |

Обе кастомные архитектуры показали критически низкое качество сегментации. Custom U-Net демонстрирует нестабильную динамику: метрики IoU и Dice колеблются около нуля, достигая локального максимума 0.036 на 8-й эпохе, после чего происходит коллапс обучения. Custom Transformer практически не отреагировала на данные, сохранив нулевые IoU/Dice на всех эпохах. Высокая Pixel Accuracy (~0.973) в обоих случаях обусловлена доминированием фона и не отражает способность модели выделять целевые объекты.

Кастомные модели уступают библиотечным аналогам на порядок:
- Максимальный IoU кастомной U-Net в 12 раз ниже, чем у библиотечной CNN-ResNet34 (0.036 против 0.437).
Custom Transformer не показала какого-либо прогресса, тогда как библиотечная MiT достигла IoU=0.102 даже при медленном старте.
- Разрыв объясняется исключительно стратегией инициализации: библиотечные модели используют веса, предобученные на ImageNet, тогда как кастомные стартуют со случайными весами.

In [ ]:
# Применение улучшений к кастомной модели
model_custom_v2 = CustomUNet(in_channels=3, classes=NUM_CLASSES)
model_custom_v2.to(device)
opt_v2 = optim.AdamW(model_custom_v2.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_v2 = optim.lr_scheduler.CosineAnnealingLR(opt_v2, T_max=40)

loss_fn_v2 = DiceLoss(mode='multiclass')

print("Training Custom U-Net + Improvements")
for ep in range(20):
    model_custom_v2.train()
    for imgs, masks in train_loader_imp: # улучшенный даталоадер с аугментациями
        imgs, masks = imgs.to(device), masks.to(device)
        opt_v2.zero_grad()
        out = model_custom_v2(imgs)
        # masks.long() гарантирует, что тензор имеет тип int64 (индексы классов)
        loss = loss_fn_v2(out, masks.long())
        loss.backward()
        opt_v2.step()
    scheduler_v2.step()

    iou_c2, dice_c2, acc_c2 = evaluate(model_custom_v2, val_loader)
    print(f"[Custom Improved] Ep {ep+1}/20 | IoU:{iou_c2:.3f} | Dice:{dice_c2:.3f} | Acc:{acc_c2:.3f}")

custom_imp_res = (iou_c2, dice_c2, acc_c2)

Training Custom U-Net + Improvements
[Custom Improved] Ep 1/20 | IoU:0.068 | Dice:0.121 | Acc:0.938
[Custom Improved] Ep 2/20 | IoU:0.090 | Dice:0.150 | Acc:0.944
[Custom Improved] Ep 3/20 | IoU:0.120 | Dice:0.199 | Acc:0.962
[Custom Improved] Ep 4/20 | IoU:0.081 | Dice:0.132 | Acc:0.971
[Custom Improved] Ep 5/20 | IoU:0.085 | Dice:0.143 | Acc:0.935
[Custom Improved] Ep 6/20 | IoU:0.165 | Dice:0.259 | Acc:0.970
[Custom Improved] Ep 7/20 | IoU:0.009 | Dice:0.018 | Acc:0.973
[Custom Improved] Ep 8/20 | IoU:0.026 | Dice:0.048 | Acc:0.974
[Custom Improved] Ep 9/20 | IoU:0.166 | Dice:0.260 | Acc:0.976
[Custom Improved] Ep 10/20 | IoU:0.000 | Dice:0.000 | Acc:0.973
[Custom Improved] Ep 11/20 | IoU:0.094 | Dice:0.160 | Acc:0.975
[Custom Improved] Ep 12/20 | IoU:0.029 | Dice:0.055 | Acc:0.973
[Custom Improved] Ep 13/20 | IoU:0.124 | Dice:0.194 | Acc:0.976
[Custom Improved] Ep 14/20 | IoU:0.191 | Dice:0.294 | Acc:0.964
[Custom Improved] Ep 15/20 | IoU:0.199 | Dice:0.306 | Acc:0.975
[Custom Impr

In [ ]:
print("Training Custom Transformer + Improvements")
model_custom_transformer.to(device)
opt_v2 = optim.AdamW(model_custom_transformer.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_v2 = optim.lr_scheduler.CosineAnnealingLR(opt_v2, T_max=40)
loss_fn_v2 = DiceLoss(mode='multiclass')

for ep in range(20):
    model_custom_transformer.train()
    for imgs, masks in train_loader_imp:
        imgs, masks = imgs.to(device), masks.to(device).long()
        opt_v2.zero_grad()
        out = model_custom_transformer(imgs)
        loss = loss_fn_v2(out, masks)
        loss.backward()
        opt_v2.step()
    scheduler_v2.step()

    iou_c2, dice_c2, acc_c2 = evaluate(model_custom_transformer, val_loader)
    print(f"[Custom Trans. Improved] Ep {ep+1}/20 | IoU:{iou_c2:.3f} | Dice:{dice_c2:.3f} | Acc:{acc_c2:.3f}")

custom_trans_imp_res = (iou_c2, dice_c2, acc_c2)

Training Custom Transformer + Improvements
[Custom Trans. Improved] Ep 1/20 | IoU:0.018 | Dice:0.035 | Acc:0.869
[Custom Trans. Improved] Ep 2/20 | IoU:0.054 | Dice:0.094 | Acc:0.901
[Custom Trans. Improved] Ep 3/20 | IoU:0.051 | Dice:0.088 | Acc:0.921
[Custom Trans. Improved] Ep 4/20 | IoU:0.057 | Dice:0.097 | Acc:0.953
[Custom Trans. Improved] Ep 5/20 | IoU:0.060 | Dice:0.101 | Acc:0.948
[Custom Trans. Improved] Ep 6/20 | IoU:0.041 | Dice:0.073 | Acc:0.956
[Custom Trans. Improved] Ep 7/20 | IoU:0.055 | Dice:0.095 | Acc:0.922
[Custom Trans. Improved] Ep 8/20 | IoU:0.060 | Dice:0.102 | Acc:0.948
[Custom Trans. Improved] Ep 9/20 | IoU:0.040 | Dice:0.071 | Acc:0.963
[Custom Trans. Improved] Ep 10/20 | IoU:0.058 | Dice:0.099 | Acc:0.955
[Custom Trans. Improved] Ep 11/20 | IoU:0.060 | Dice:0.101 | Acc:0.948
[Custom Trans. Improved] Ep 12/20 | IoU:0.058 | Dice:0.099 | Acc:0.942
[Custom Trans. Improved] Ep 13/20 | IoU:0.057 | Dice:0.097 | Acc:0.958
[Custom Trans. Improved] Ep 14/20 | IoU:0.0

Выводы по обучению кастомных моделей с улучшениями:

| Модель | Режим | Эпохи | IoU (final) | Dice (final) | Acc (final) | Стабильность обучения |
|-|-|-|-|-|-|-|
| Custom U-Net + Improvements | Custom Improved | 20 | 0.243 | 0.356 | 0.977 | Низкая, резкие скачки |
| Custom Transformer + Improvements | Custom Improved | 20 | 0.060 | 0.102 | 0.951 | Высокая, но плато на низких метриках |
| DeepLabV3+ + MiT | Lib Improved | 20 | 0.116 | 0.196 | 0.957 | Средняя, колебания |
| SegFormer + Improvements | Lib Improved | 20 | 0.255 | 0.368 | 0.977 | Высокая, устойчивый рост |

|Сравнение | Результат | Вывод |
|-|-|-|
|Custom U-Net Imp и Lib DLv3+ Imp | IoU: 0.243 и 0.116|  Простая кастомная архитектура с улучшениями превзошла сложную библиотечную на малых данных |
|Custom Trans Imp и Lib SegFormer Imp | IoU: 0.060 и 0.255 | Разрыв в 4 раза подтверждает критическую важность предобученных весов для Transformer|
|Лучшая кастомная (U-Net) и Лучшая библиотечная (SegFormer) | 0.243 и 0.255 | Кастомная реализация при грамотных улучшениях может догнать SOTA-библиотеки|

Оценка качества улучшенных кастомных моделей:
- Custom U-Net + Improvements: IoU=0.243, Dice=0.356. Применение аугментаций, DiceLoss и планировщика позволило модели выйти на конкурентный уровень. Однако обучение остаётся нестабильным: наблюдаются резкие провалы метрик (например, эпохи 6-7: IoU падает с 0.165 до 0.009), что указывает на чувствительность к шуму в градиентах и необходимость более тщательного подбора learning rate.
- Custom Transformer + Improvements: IoU=0.060, Dice=0.102. Модель быстро вышла на плато и не показала прогресса после 4 эпохи. Это характерно для архитектур с механизмом внимания, обучаемых с нуля: без предобученных весов они не могут эффективно извлекать признаки из малых данных даже при использовании продвинутых техник регуляризации.